# Week 6 — Model Evaluation, Hyperparameter Tuning & Final Model Selection

**Project:** MaintAI — Predictive Maintenance & RUL Prediction for Turbofan Engines  

This notebook:
1. Loads all four previously trained model families
2. Performs comprehensive evaluation: RMSE, MAE, R²
3. Runs hyperparameter tuning (RandomizedSearchCV) on the best classical model
4. Produces a unified model comparison table and charts
5. Selects and saves the **final deployment model**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import metrics
import xgboost as xgb

print('Libraries loaded.')

## 1. Data Loading & Preprocessing

In [ ]:
data_dir = os.path.join('..', 'Dataset')

SELECTED_SENSORS = ['Cycle',
                    'SensorMeasure2','SensorMeasure3','SensorMeasure4',
                    'SensorMeasure7','SensorMeasure8','SensorMeasure11',
                    'SensorMeasure12','SensorMeasure13','SensorMeasure15',
                    'SensorMeasure17','SensorMeasure20','SensorMeasure21']

def load_and_preprocess(subset_num, rul_cap=125):
    path = os.path.join(data_dir, f'train_FD00{subset_num}.txt')
    cols = ['ID', 'Cycle', 'OpSet1', 'OpSet2', 'OpSet3'] + \
           [f'SensorMeasure{i}' for i in range(1, 22)]
    df = pd.read_csv(path, sep=' ', header=None)
    df.drop(df.columns[[26, 27]], axis=1, inplace=True)
    df.columns = cols
    max_c = df.groupby('ID')['Cycle'].max().reset_index()
    max_c.columns = ['ID', 'MaxCycle']
    df = df.merge(max_c, on='ID')
    df['RUL'] = (df['MaxCycle'] - df['Cycle']).clip(upper=rul_cap)
    df.drop('MaxCycle', axis=1, inplace=True)
    return df

df = pd.concat([load_and_preprocess(i) for i in range(1, 5)], ignore_index=True)

X = df[SELECTED_SENSORS]
y = df['RUL']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Evaluation Helper

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, scaled=False):
    """Fit model (or use pre-fitted), evaluate on train+test."""
    if scaled:
        p_tr = model.predict(X_tr)
        p_te = model.predict(X_te)
    else:
        p_tr = model.predict(X_tr)
        p_te = model.predict(X_te)
    
    def m(y_true, y_pred):
        return {
            'RMSE': np.sqrt(metrics.mean_squared_error(y_true, y_pred)),
            'MAE' : metrics.mean_absolute_error(y_true, y_pred),
            'R2'  : metrics.r2_score(y_true, y_pred)
        }
    tr = m(y_tr, p_tr)
    te = m(y_te, p_te)
    print(f"{name:30s}  Test RMSE={te['RMSE']:.2f}  MAE={te['MAE']:.2f}  R²={te['R2']:.4f}")
    return tr, te, p_te

results = {}

## 3. Train & Evaluate All Models

In [ ]:
# ── 3a. Linear Regression ──────────────────────────────────────────────────────
lr = LinearRegression().fit(X_train_sc, y_train)
tr, te, pred_lr = evaluate('Linear Regression', lr, X_train_sc, y_train, X_test_sc, y_test, scaled=True)
results['Linear Regression'] = {'Train': tr, 'Test': te}

In [ ]:
# ── 3b. Random Forest (default) ───────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
tr, te, pred_rf = evaluate('Random Forest (default)', rf, X_train, y_train, X_test, y_test)
results['Random Forest'] = {'Train': tr, 'Test': te}

In [ ]:
# ── 3c. XGBoost ───────────────────────────────────────────────────────────────
xgb_m = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=42, n_jobs=-1, verbosity=0)
xgb_m.fit(X_train, y_train)
tr, te, pred_xgb = evaluate('XGBoost', xgb_m, X_train, y_train, X_test, y_test)
results['XGBoost'] = {'Train': tr, 'Test': te}

In [ ]:
# ── 3d. Gradient Boosting (sklearn) ──────────────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                subsample=0.8, random_state=42)
gb.fit(X_train, y_train)
tr, te, pred_gb = evaluate('Gradient Boosting', gb, X_train, y_train, X_test, y_test)
results['Gradient Boosting'] = {'Train': tr, 'Test': te}

## 4. Hyperparameter Tuning — XGBoost (RandomizedSearchCV)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'    : [200, 400, 600],
    'max_depth'       : [4, 5, 6, 7, 8],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.15],
    'subsample'       : [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma'           : [0, 0.1, 0.3],
}

xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)

rscv = RandomizedSearchCV(
    xgb_base, param_dist,
    n_iter      = 30,
    cv          = 3,
    scoring     = 'neg_root_mean_squared_error',
    random_state= 42,
    n_jobs      = -1,
    verbose     = 1
)
rscv.fit(X_train, y_train)

print('\nBest params:', rscv.best_params_)
print('Best CV RMSE:', -rscv.best_score_)

In [ ]:
# Evaluate tuned XGBoost
best_xgb = rscv.best_estimator_
tr, te, pred_xgb_tuned = evaluate('XGBoost (Tuned)', best_xgb, X_train, y_train, X_test, y_test)
results['XGBoost (Tuned)'] = {'Train': tr, 'Test': te}

## 5. Unified Comparison Table

In [ ]:
rows = []
for m_name, res in results.items():
    rows.append({
        'Model'      : m_name,
        'Train RMSE' : round(res['Train']['RMSE'], 2),
        'Test RMSE'  : round(res['Test']['RMSE'],  2),
        'Train MAE'  : round(res['Train']['MAE'],  2),
        'Test MAE'   : round(res['Test']['MAE'],   2),
        'Train R²'   : round(res['Train']['R2'],   4),
        'Test R²'    : round(res['Test']['R2'],    4),
        'Overfit (RMSE gap)': round(res['Test']['RMSE'] - res['Train']['RMSE'], 2)
    })

comp = pd.DataFrame(rows).sort_values('Test RMSE')
print('\n' + '='*80)
print('           FINAL MODEL COMPARISON TABLE — Week 6')
print('='*80)
print(comp.to_string(index=False))
comp

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = 'Set2'

for ax, (col, lbl) in zip(axes, [
    ('Test RMSE', 'RMSE ↓ (lower = better)'),
    ('Test MAE',  'MAE ↓ (lower = better)'),
    ('Test R²',   'R² ↑ (higher = better)'),
]):
    bars = ax.bar(comp['Model'], comp[col], color=sns.color_palette(palette, len(comp)))
    ax.set_title(lbl, fontsize=12)
    ax.set_xticklabels(comp['Model'], rotation=20, ha='right', fontsize=9)
    for bar, val in zip(bars, comp[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Week 6 — Model Evaluation Summary', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Actual vs Predicted — Best Model

In [ ]:
best_pred = best_xgb.predict(X_test)
n = 600

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.values[:n], label='Actual RUL', color='steelblue', lw=1.5)
ax.plot(best_pred[:n],     label='Predicted RUL (Tuned XGBoost)', color='darkorange', alpha=0.8, lw=1.5)
ax.fill_between(range(n),
                y_test.values[:n], best_pred[:n],
                alpha=0.15, color='gray', label='Error area')
ax.set_title('Actual vs Predicted RUL — Tuned XGBoost (Test Set)', fontsize=13)
ax.set_xlabel('Sample Index'); ax.set_ylabel('RUL (cycles)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Residual plot
residuals = y_test.values - best_pred
plt.figure(figsize=(10, 4))
plt.scatter(best_pred, residuals, alpha=0.3, s=10, color='purple')
plt.axhline(0, color='red', lw=1.5, linestyle='--')
plt.title('Residual Plot — Tuned XGBoost')
plt.xlabel('Predicted RUL'); plt.ylabel('Residual (Actual - Predicted)')
plt.tight_layout(); plt.show()

## 7. Save Final Model for Deployment

In [ ]:
model_dir = os.path.join('..', 'models')
os.makedirs(model_dir, exist_ok=True)

# Final deployment model = Tuned XGBoost
joblib.dump(best_xgb, os.path.join(model_dir, 'final_model.pkl'))
joblib.dump(scaler,   os.path.join(model_dir, 'scaler.pkl'))

with open(os.path.join(model_dir, 'feature_list.json'), 'w') as f:
    json.dump(SELECTED_SENSORS, f)

# Save comparison table
comp.to_csv(os.path.join(model_dir, 'model_comparison.csv'), index=False)

print('=== Final model saved successfully ===')
print(f'Path: {os.path.abspath(model_dir)}')
print('Best model: Tuned XGBoost')
print(f"Test RMSE: {results['XGBoost (Tuned)']['Test']['RMSE']:.2f}")
print(f"Test MAE:  {results['XGBoost (Tuned)']['Test']['MAE']:.2f}")
print(f"Test R²:   {results['XGBoost (Tuned)']['Test']['R2']:.4f}")